# H7 · "Sales matters just as much as product" (*Zero to One*, ch. 2 and 11)

Thiel lists "focus on product, not sales" among the false lessons of the dot-com crash, and argues the opposite: "sales matters just as much as product". Engineers, he says, underrate distribution. A business works only if customer lifetime value exceeds the cost of acquiring the customer, and "it takes hard work to make sales look easy".

| Claim | Test | Data |
|---|---|---|
| **H7a.** Successful technology firms spend heavily on distribution, not only on engineering. | Sales-and-marketing spending against R&D spending, as shares of revenue, for software and internet firms and for all other firms that report both, 2011–2025. | SEC XBRL `SellingAndMarketingExpense`, `ResearchAndDevelopmentExpense` |
| **H7b.** Distribution spending pays: it buys growth. | Next-year revenue growth on this year's sales-and-marketing and R&D intensity, comparing firms in the same industry and year. Thiel predicts a sales coefficient at least as large as the R&D one. | Same, plus revenue |
| **H7c.** Has selling become harder since the book? | "Sales efficiency" (the SaaS "magic number"): the increase in revenue for each dollar of the previous year's sales-and-marketing spending, by year. | Same |
| **H7d.** Investors value distribution. | Firm value (public float ÷ revenue) on the two intensities, in the same industry and year, with growth and margin held fixed. | Same, plus public float |

**Caveats**
* Only firms that report sales and marketing as a separate line can be tested; most are technology, healthcare and consumer firms. Others report it inside SG&A, which also contains administrative costs.
* Spending is a choice, not a random treatment: firms with better products may spend more on sales because it pays them to. The regressions describe association, not cause.
* Public float again stands in for market value (see H1).

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
import sys; from pathlib import Path  # thiel_common.py lives in the repo root
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "thiel_common.py").exists())))
import thiel_common as tc
tc.style()
print("synthetic mode" if tc.SYNTHETIC else "live data")
YEARS = range(2011, 2026)
fin = tc.financial_panel(YEARS)
sm = tc.annual_tag("SellingAndMarketingExpense", YEARS, "sm")
rd = tc.annual_tag("ResearchAndDevelopmentExpense", YEARS, "rd")
sga = tc.annual_tag("SellingGeneralAndAdministrativeExpense", YEARS, "sga")
d = fin.merge(sm, on=["cik", "year"], how="left").merge(rd, on=["cik", "year"], how="left").merge(sga, on=["cik", "year"], how="left")
s4 = d.sic.astype(int)
d["software"] = s4.between(7370, 7379)
d["sector"] = np.select([s4.between(7370, 7379), s4.between(2833, 2836) | s4.between(3841, 3851) | s4.between(8000, 8099),
                         s4.between(3570, 3579) | s4.between(3660, 3679)],
                        ["Software and internet", "Pharma, biotech, medical", "Hardware and electronics"], "Other")
d["sm_int"] = d.sm / d.revenue
d["rd_int"] = d.rd / d.revenue
both = d[(d.sm > 0) & (d.rd > 0) & d.sm_int.between(0, 3) & d.rd_int.between(0, 3)].copy()
print(f"{len(d):,} firm-years in the panel; {d.sm.notna().sum():,} report sales and marketing, {d.rd.notna().sum():,} report R&D; "
      f"{len(both):,} report both ({both.cik.nunique():,} firms)")
both.groupby("sector").agg(firm_years=("cik", "size"), firms=("cik", "nunique"))

## H7a · Product or distribution: where does the money go?

In [ ]:
med = both.groupby(["sector", "year"])[["sm_int", "rd_int"]].median().mul(100)
share_sm_gt_rd = both.assign(sm_gt=both.sm > both.rd).groupby(["sector", "year"]).sm_gt.mean().mul(100)
tab = med.join(share_sm_gt_rd.rename("pct_firms_sm_above_rd")).round(1)
tc.save_table(tab, "h7a_intensity_by_sector_year")
by_sector = both.groupby("sector").agg(median_sm_pct=("sm_int", lambda x: 100 * x.median()), median_rd_pct=("rd_int", lambda x: 100 * x.median()),
                                       pct_firms_sm_above_rd=("sm", lambda x: 100 * (x > both.loc[x.index, "rd"]).mean()))
sw = both[both.software]
big = sw.groupby("cik").revenue.max() >= 1e9
sw_big = sw[sw.cik.isin(big[big].index)]
print("Software firms with revenue of $1bn or more (in any year):",
      f"median S&M {100 * sw_big.sm_int.median():.1f}% of revenue, R&D {100 * sw_big.rd_int.median():.1f}%; "
      f"S&M above R&D in {100 * (sw_big.sm > sw_big.rd).mean():.0f}% of firm-years")
by_sector.round(1)

In [ ]:
sectors = ["Software and internet", "Pharma, biotech, medical", "Hardware and electronics", "Other"]
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4), sharey=True)
present = [x for x in ["Software and internet", "Pharma, biotech, medical", "Hardware and electronics", "Other"] if x in med.index.get_level_values(0)][:2]
for ax, sec in zip(axes, present):
    m = med.loc[sec]
    ax.plot(m.index, m.sm_int, color=tc.C[1], marker="o", ms=4, lw=2, label="Sales and marketing")
    ax.plot(m.index, m.rd_int, color=tc.C[0], marker="o", ms=4, lw=2, label="R&D")
    ax.axvline(2014.7, color=tc.GREY, lw=0.8)
    ax.set_title(f"{sec}: median % of revenue", loc="left"); ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2)
axes[0].set_ylabel("% of revenue (median firm)")
fig.tight_layout(); tc.save_fig(fig, "h7_fig1_intensity")

## H7b · Does distribution spending buy growth?

* **Outcome:** next year's revenue growth, log(revenue at t+1 ÷ revenue at t).
* **Explanatory variables:** this year's sales-and-marketing intensity and R&D intensity, plus firm size (log revenue).
* **Comparison:** firms in the same 4-digit SIC industry and year (industry × year fixed effects), with standard errors clustered by firm.
* **Pre and post:** separate estimates for 2011–2014 and 2015 onward.

In [ ]:
both = both.sort_values(["cik", "year"])
nxt = d.set_index(["cik", "year"]).revenue
both["rev_next"] = [nxt.get((c, y + 1), np.nan) for c, y in zip(both.cik, both.year)]
both["g_next"] = np.log(both.rev_next / both.revenue)
both["log_rev"] = np.log(both.revenue)
both["ind_year"] = both.sic + "_" + both.year.astype(str)
g = both[both.g_next.between(-2, 2) & both.sm_int.between(0, 2) & both.rd_int.between(0, 2)]
rows = []
for lab, sub in [("all", g), ("software and internet", g[g.software]),
                 ("all, 2011-2014", g[g.year <= 2014]), ("all, 2015+", g[g.year >= 2015])]:
    t = tc.fe_ols(sub, "g_next", ["sm_int", "rd_int", "log_rev"], ["ind_year"], "cik")
    for x in ["sm_int", "rd_int"]:
        rows.append({"sample": lab, "x": x, **t.loc[x, ["coef", "se", "p", "n", "clusters"]].to_dict()})
gr = pd.DataFrame(rows)
for c in ["coef", "se", "p", "n", "clusters"]: gr[c] = pd.to_numeric(gr[c])
tc.save_table(gr, "h7b_growth_regressions", index=False)
gr.round(4)

## H7c · Has selling become harder since the book?

Sales efficiency = (revenue at t − revenue at t−1) ÷ sales-and-marketing spending at t−1. A value of 1 means each dollar of last year's sales spending was matched by a dollar of new revenue. Medians for software and internet firms, by year.

In [ ]:
prev = d.set_index(["cik", "year"])
both["rev_prev"] = [prev.revenue.get((c, y - 1), np.nan) for c, y in zip(both.cik, both.year)]
both["sm_prev"] = [prev.sm.get((c, y - 1), np.nan) for c, y in zip(both.cik, both.year)]
both["efficiency"] = (both.revenue - both.rev_prev) / both.sm_prev
eff = both[both.software & both.sm_prev.gt(0)].copy()
eff = eff[eff.efficiency.between(-5, 10)]
eff_y = eff.groupby("year").efficiency.agg(["median", "size"]).rename(columns={"size": "n"})
eff_big = eff[eff.cik.isin(big[big].index)].groupby("year").efficiency.median().rename("median_rev_1bn_plus")
eff_y = eff_y.join(eff_big)
tc.save_table(eff_y, "h7c_sales_efficiency")
fig, ax = plt.subplots(figsize=(8.5, 3.8))
ax.plot(eff_y.index, eff_y["median"], color=tc.C[0], marker="o", ms=5, lw=2, label="All software and internet firms")
ax.plot(eff_y.index, eff_y.median_rev_1bn_plus, color=tc.C[1], marker="o", ms=5, lw=2, label="Firms with $1bn+ revenue")
ax.axhline(0, color=tc.INK, lw=0.6); ax.axvline(2014.7, color=tc.GREY, lw=0.8)
ax.set_ylabel("New revenue per $1 of prior-year S&M (median)"); ax.legend(loc="lower left")
ax.set_title("Sales efficiency of software and internet firms", loc="left")
fig.tight_layout(); tc.save_fig(fig, "h7_fig2_efficiency")
eff_y.round(2)

## H7d · Do investors value distribution?

log(public float ÷ revenue) on the two intensities, with revenue growth and operating margin held fixed, comparing firms in the same industry and year.

In [ ]:
pf = tc.public_float_panel(YEARS)
pfv = pf.set_index(["cik", "year"]).val
v = both.copy()
v["float"] = [pfv.get((c, y), np.nan) for c, y in zip(v.cik, v.year)]
v["log_ps"] = np.log(v["float"] / v.revenue)
v["g_now"] = np.log(v.revenue / v.rev_prev)
v = v.replace([np.inf, -np.inf], np.nan).dropna(subset=["log_ps", "g_now", "op_margin"])
v = v[v.g_now.between(-2, 2) & v.sm_int.between(0, 2) & v.rd_int.between(0, 2)]
rows = []
for lab, sub in [("all", v), ("software and internet", v[v.software])]:
    t = tc.fe_ols(sub, "log_ps", ["sm_int", "rd_int", "g_now", "op_margin"], ["ind_year"], "cik")
    for x in ["sm_int", "rd_int", "g_now", "op_margin"]:
        rows.append({"sample": lab, "x": x, **t.loc[x, ["coef", "se", "p", "n", "clusters"]].to_dict()})
vr = pd.DataFrame(rows)
for c in ["coef", "se", "p", "n", "clusters"]: vr[c] = pd.to_numeric(vr[c])
tc.save_table(vr, "h7d_valuation_regressions", index=False)
vr.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8))
lab = {"sm_int": "Sales and marketing", "rd_int": "R&D"}
for ax, (title, tab_, samples, scale, unit) in zip(axes, [
        ("Next-year revenue growth", gr, ["all", "software and internet", "all, 2011-2014", "all, 2015+"], 0.1, "log points of growth"),
        ("Value (float / revenue)", vr[vr.x.isin(["sm_int", "rd_int"])], ["all", "software and internet"], 0.1, "log points of value")]):
    y = np.arange(len(samples))
    for i, x in enumerate(["sm_int", "rd_int"]):
        t = tab_[tab_.x == x].set_index("sample").reindex(samples)
        ax.errorbar(t.coef * scale, y + (i - 0.5) * 0.25, xerr=1.96 * t.se * scale, fmt="o", color=tc.C[1 - i], ms=6, capsize=3, label=lab[x])
    ax.axvline(0, color=tc.INK, lw=0.7); ax.set_yticks(y); ax.set_yticklabels(samples); ax.invert_yaxis()
    ax.set_xlabel(f"Effect of spending 10 more points of revenue ({unit})")
    ax.set_title(title, loc="left"); ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2, fontsize=8)
fig.tight_layout(); tc.save_fig(fig, "h7_fig3_coefficients")

## Verdict inputs and check cell — send me this output

In [ ]:
rec = lambda df: json.loads(df.round(4).to_json(orient="records"))
summary = {
    "n_both": int(len(both)), "firms_both": int(both.cik.nunique()),
    "by_sector": json.loads(by_sector.round(2).to_json(orient="index")),
    "software_1bn": {"median_sm_pct": round(100 * float(sw_big.sm_int.median()), 2), "median_rd_pct": round(100 * float(sw_big.rd_int.median()), 2),
                     "pct_sm_above_rd": round(100 * float((sw_big.sm > sw_big.rd).mean()), 1), "firms": int(sw_big.cik.nunique())},
    "software_median_by_year": json.loads(med.loc["Software and internet"].round(2).to_json(orient="index")),
    "growth_regressions": rec(gr), "valuation_regressions": rec(vr),
    "efficiency": json.loads(eff_y.round(3).to_json(orient="index")),
}
(tc.OUT / "h7_summary.json").write_text(json.dumps(summary, indent=1, default=str))
print(json.dumps(summary, indent=1, default=str)[:5000])